# 🎨 Taller - Conversión y Manipulación de Espacios de Color

**Objetivo:** Trabajar con diferentes espacios de color (RGB, HSL, HSV, LAB, CIE XYZ), realizar conversiones entre ellos y aplicarlos en manipulación de imágenes, color grading, segmentación por color y creación de paletas armónicas.

**Herramientas:** `opencv-python`, `scikit-image`, `numpy`, `matplotlib`, `colormath`, `sklearn`

---
**Contenido:**
1. Instalación y configuración
2. Carga de imagen
3. Conversión entre espacios de color
4. Visualización 3D de espacios de color
5. Segmentación por color en HSV
6. Manipulación de color
7. Color Grading — LUTs y filtros cinemáticos
8. Paletas de colores con K-means
9. Análisis de histogramas
10. Bonus — Corrección automática, daltonismo y transfer de color

## 📦 Sección 1 — Instalación y configuración

In [ ]:
!pip install opencv-python scikit-image numpy matplotlib colormath scikit-learn --quiet
print('✅ Dependencias instaladas.')

In [ ]:
# ══════════════════════════════════════════════════════════
# CONFIGURACIÓN — ajusta esta variable con el nombre
# de tu imagen .jpg subida a Colab
# ══════════════════════════════════════════════════════════
IMAGEN_BASE = 'imagen.jpg'   # <-- cambia esto por tu archivo
print(f'Imagen configurada: {IMAGEN_BASE}')

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
from skimage import color as skcolor, exposure
from skimage.util import img_as_float, img_as_ubyte
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor' : '#0f1117',
    'axes.facecolor'   : '#1a1d27',
    'axes.edgecolor'   : '#3a3f55',
    'axes.labelcolor'  : '#c8cce0',
    'xtick.color'      : '#6a6f8a',
    'ytick.color'      : '#6a6f8a',
    'text.color'       : '#c8cce0',
    'grid.color'       : '#2a2f45',
    'grid.alpha'       : 0.5,
    'font.family'      : 'monospace',
})
print('✅ Importaciones listas.')

## 🖼️ Sección 2 — Carga de imagen

Sube tu imagen `.jpg` usando el panel de archivos de Colab (ícono de carpeta a la izquierda).

In [ ]:
img_bgr = cv2.imread(IMAGEN_BASE)
if img_bgr is None:
    raise FileNotFoundError(f'No se encontró {IMAGEN_BASE}. Súbela a Colab.')

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

MAX_DIM = 600
h, w = img_rgb.shape[:2]
if max(h, w) > MAX_DIM:
    escala = MAX_DIM / max(h, w)
    img_rgb = cv2.resize(img_rgb, (int(w*escala), int(h*escala)))
    print(f'Imagen redimensionada a {img_rgb.shape[1]}x{img_rgb.shape[0]}')

h, w = img_rgb.shape[:2]
print(f'Imagen cargada: {w}x{h} px  |  dtype: {img_rgb.dtype}')

plt.figure(figsize=(6, 4))
plt.imshow(img_rgb)
plt.title('Imagen original (RGB)', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.savefig('imagen_original.png', dpi=100, bbox_inches='tight')
plt.show()

## 🔄 Sección 3 — Conversión entre espacios de color

| Espacio | Canales | Rango típico | Uso principal |
|---|---|---|---|
| RGB | R, G, B | 0-255 | Visualización en pantalla |
| HSV | H, S, V | H:0-180, S:0-255, V:0-255 | Segmentación |
| HLS | H, L, S | H:0-180, L:0-255, S:0-255 | Edición de color |
| LAB | L, a, b | L:0-100, a/b:-128-127 | Percepción uniforme |
| YCrCb | Y, Cr, Cb | 0-255 | Compresión de video |
| XYZ | X, Y, Z | 0-1 | Espacio de referencia CIE |

In [ ]:
img_hsv   = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
img_hls   = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HLS)
img_lab   = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
img_ycrcb = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2YCrCb)
img_gray  = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
img_xyz   = skcolor.rgb2xyz(img_rgb / 255.0)

print('Rangos de valores por espacio de color:')
print('=' * 55)
espacios = {'RGB': img_rgb, 'HSV': img_hsv, 'HLS': img_hls, 'LAB': img_lab, 'YCrCb': img_ycrcb}
for nombre, img in espacios.items():
    for i, canal in enumerate(cv2.split(img)):
        print(f'   {nombre} canal {i}: min={canal.min():6.1f}  max={canal.max():6.1f}  mean={canal.mean():6.1f}')
    print()

In [ ]:
def mostrar_canales(imagen, nombres_canales, titulo, cmap_list=None):
    canales = cv2.split(imagen)
    fig, axes = plt.subplots(1, len(canales)+1, figsize=(4*(len(canales)+1), 3.5))
    axes[0].imshow(img_rgb)
    axes[0].set_title('Original', fontsize=10)
    axes[0].axis('off')
    for i, (canal, nombre) in enumerate(zip(canales, nombres_canales)):
        cmap = cmap_list[i] if cmap_list else 'gray'
        axes[i+1].imshow(canal, cmap=cmap, vmin=0, vmax=canal.max())
        axes[i+1].set_title(f'{nombre}\n[{canal.min():.0f}-{canal.max():.0f}]', fontsize=9)
        axes[i+1].axis('off')
    fig.suptitle(titulo, fontsize=13, y=1.02)
    plt.tight_layout()
    return fig

mostrar_canales(img_rgb, ['R','G','B'], 'RGB - Canales', ['Reds','Greens','Blues'])
plt.savefig('canales_rgb.png', dpi=100, bbox_inches='tight'); plt.show()

mostrar_canales(img_hsv, ['H (Matiz)','S (Saturacion)','V (Valor)'], 'HSV - Canales')
plt.savefig('canales_hsv.png', dpi=100, bbox_inches='tight'); plt.show()

mostrar_canales(img_lab, ['L (Luminosidad)','a (verde-rojo)','b (azul-amarillo)'], 'LAB - Canales')
plt.savefig('canales_lab.png', dpi=100, bbox_inches='tight'); plt.show()

mostrar_canales(img_ycrcb, ['Y (Luma)','Cr','Cb'], 'YCrCb - Canales')
plt.savefig('canales_ycrcb.png', dpi=100, bbox_inches='tight'); plt.show()
print('Canales visualizados.')

## 🌐 Sección 4 — Visualización 3D de espacios de color

In [ ]:
pixels = img_rgb.reshape(-1, 3)
n_muestra = min(3000, len(pixels))
idx = np.random.choice(len(pixels), n_muestra, replace=False)
muestra = pixels[idx]
colores_norm = muestra / 255.0

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.set_facecolor('#1a1d27')
ax1.scatter(muestra[:,0], muestra[:,1], muestra[:,2], c=colores_norm, s=3, alpha=0.6)
ax1.set_xlabel('R'); ax1.set_ylabel('G'); ax1.set_zlabel('B')
ax1.set_title('Espacio RGB 3D', fontsize=11)

hsv_muestra = cv2.cvtColor(muestra.reshape(1,-1,3), cv2.COLOR_RGB2HSV).reshape(-1,3)
H = hsv_muestra[:,0] / 180.0 * np.pi * 2
S = hsv_muestra[:,1] / 255.0
V = hsv_muestra[:,2] / 255.0
X_cil = S * np.cos(H)
Y_cil = S * np.sin(H)

ax2 = fig.add_subplot(122, projection='3d')
ax2.set_facecolor('#1a1d27')
ax2.scatter(X_cil, Y_cil, V, c=colores_norm, s=3, alpha=0.6)
ax2.set_xlabel('S*cos(H)'); ax2.set_ylabel('S*sin(H)'); ax2.set_zlabel('V')
ax2.set_title('Espacio HSV 3D (cilindrico)', fontsize=11)

fig.suptitle('Distribucion de colores en 3D', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('espacios_3d.png', dpi=100, bbox_inches='tight')
plt.show()
print('Visualizacion 3D generada.')

## 🎯 Sección 5 — Segmentación por color en HSV

In [ ]:
rangos_color = {
    'Rojo'    : [(  0, 50, 50), ( 10, 255, 255)],
    'Rojo2'   : [(170, 50, 50), (180, 255, 255)],
    'Azul'    : [(100, 50, 50), (130, 255, 255)],
    'Verde'   : [( 40, 50, 50), ( 80, 255, 255)],
    'Amarillo': [( 20, 50, 50), ( 35, 255, 255)],
}

img_hsv_seg = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

mascaras = {}
for nombre, (lower, upper) in rangos_color.items():
    m = cv2.inRange(img_hsv_seg, np.array(lower), np.array(upper))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, kernel)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel)
    mascaras[nombre] = m

mascaras['Rojo_total'] = cv2.bitwise_or(mascaras['Rojo'], mascaras['Rojo2'])

colores_viz = ['Rojo_total', 'Azul', 'Verde', 'Amarillo']
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, nombre in enumerate(colores_viz):
    mascara = mascaras[nombre]
    objeto = cv2.bitwise_and(img_rgb, img_rgb, mask=mascara)
    axes[0,i].imshow(mascara, cmap='gray')
    axes[0,i].set_title(f'Mascara {nombre}', fontsize=10)
    axes[0,i].axis('off')
    axes[1,i].imshow(objeto)
    axes[1,i].set_title(f'Segmentado {nombre}', fontsize=10)
    axes[1,i].axis('off')

fig.suptitle('Segmentacion por color en HSV', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('segmentacion_color.png', dpi=100, bbox_inches='tight')
plt.show()
print('Segmentacion completada.')

## 🎛️ Sección 6 — Manipulación de color

In [ ]:
def ajustar_saturacion(img_rgb, factor):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[:,:,1] = np.clip(hsv[:,:,1] * factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

def rotar_matiz(img_rgb, grados):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.int32)
    hsv[:,:,0] = (hsv[:,:,0] + grados) % 180
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

def ajustar_luminosidad_lab(img_rgb, delta):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    lab[:,:,0] = np.clip(lab[:,:,0] + delta, 0, 255)
    return cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2RGB)

def balance_blancos(img_rgb):
    resultado = img_rgb.astype(np.float32)
    media_global = resultado.mean()
    for i in range(3):
        media_canal = resultado[:,:,i].mean()
        if media_canal > 0:
            resultado[:,:,i] *= media_global / media_canal
    return np.clip(resultado, 0, 255).astype(np.uint8)

resultados = {
    'Original'             : img_rgb,
    'Saturacion +50%'      : ajustar_saturacion(img_rgb, 1.5),
    'Saturacion -50%'      : ajustar_saturacion(img_rgb, 0.5),
    'Matiz +60 grados'     : rotar_matiz(img_rgb, 60),
    'Matiz +120 grados'    : rotar_matiz(img_rgb, 120),
    'Luminosidad +40 LAB'  : ajustar_luminosidad_lab(img_rgb, 40),
    'Luminosidad -40 LAB'  : ajustar_luminosidad_lab(img_rgb, -40),
    'Balance de blancos'   : balance_blancos(img_rgb),
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (titulo, img) in zip(axes.flatten(), resultados.items()):
    ax.imshow(img)
    ax.set_title(titulo, fontsize=9)
    ax.axis('off')
fig.suptitle('Manipulacion de color en distintos espacios', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('manipulacion_color.png', dpi=100, bbox_inches='tight')
plt.show()
print('Manipulaciones aplicadas.')

## 🎬 Sección 7 — Color Grading: LUTs y filtros cinematicos

In [ ]:
def aplicar_lut(img, lut_r, lut_g, lut_b):
    r = np.zeros_like(img)
    r[:,:,0] = lut_r[img[:,:,0]]
    r[:,:,1] = lut_g[img[:,:,1]]
    r[:,:,2] = lut_b[img[:,:,2]]
    return r

x = np.arange(256)

lut_r_warm = np.clip(x*1.2,  0,255).astype(np.uint8)
lut_g_warm = np.clip(x*1.05, 0,255).astype(np.uint8)
lut_b_warm = np.clip(x*0.8,  0,255).astype(np.uint8)
filtro_warm = aplicar_lut(img_rgb, lut_r_warm, lut_g_warm, lut_b_warm)

lut_r_cool = np.clip(x*0.8,  0,255).astype(np.uint8)
lut_g_cool = np.clip(x*0.95, 0,255).astype(np.uint8)
lut_b_cool = np.clip(x*1.2,  0,255).astype(np.uint8)
filtro_cool = aplicar_lut(img_rgb, lut_r_cool, lut_g_cool, lut_b_cool)

lut_fade = np.clip(x*0.8+30, 0,255).astype(np.uint8)
filtro_fade = aplicar_lut(img_rgb, lut_fade, lut_fade, lut_fade)

lut_r_cross = np.clip(x*1.3-20, 0,255).astype(np.uint8)
lut_g_cross = np.clip(x*0.9+10, 0,255).astype(np.uint8)
lut_b_cross = np.clip(x*0.7+40, 0,255).astype(np.uint8)
filtro_cross = aplicar_lut(img_rgb, lut_r_cross, lut_g_cross, lut_b_cross)

gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
lut_noir = np.clip(((x/255.0)**0.7)*255, 0,255).astype(np.uint8)
filtro_noir = cv2.cvtColor(lut_noir[gray], cv2.COLOR_GRAY2RGB)

def curva_s(x, intensidad=1.5):
    t = x/255.0
    s = t + intensidad*t*(1-t)*(t-0.5)
    return np.clip(s*255, 0,255).astype(np.uint8)
lut_s = curva_s(x)
filtro_curva_s = aplicar_lut(img_rgb, lut_s, lut_s, lut_s)

filtros = {
    'Original'     : img_rgb,
    'Warm'         : filtro_warm,
    'Cool'         : filtro_cool,
    'Fade'         : filtro_fade,
    'Cross Process': filtro_cross,
    'Noir'         : filtro_noir,
    'Curva S'      : filtro_curva_s,
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (titulo, img) in zip(axes.flatten(), filtros.items()):
    ax.imshow(img); ax.set_title(titulo, fontsize=10); ax.axis('off')
axes.flatten()[-1].axis('off')
fig.suptitle('Color Grading - LUTs y filtros cinematicos', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('color_grading.png', dpi=100, bbox_inches='tight')
plt.show()
print('Color grading completado.')

## 🎨 Sección 8 — Paletas de colores con K-means

In [ ]:
N_COLORES = 8
pixels_flat = img_rgb.reshape(-1, 3).astype(np.float32)
kmeans = KMeans(n_clusters=N_COLORES, random_state=42, n_init=10)
kmeans.fit(pixels_flat)
colores_dominantes = kmeans.cluster_centers_.astype(np.uint8)
etiquetas = kmeans.labels_
conteos = np.bincount(etiquetas)
porcentajes = conteos / conteos.sum() * 100
orden = np.argsort(porcentajes)[::-1]
colores_ord = colores_dominantes[orden]
porcentajes_ord = porcentajes[orden]

print('Colores dominantes:')
for i, (c, p) in enumerate(zip(colores_ord, porcentajes_ord)):
    print(f'  Color {i+1}: RGB({c[0]},{c[1]},{c[2]})  #{c[0]:02x}{c[1]:02x}{c[2]:02x}  {p:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
pos = 0
for c, p in zip(colores_ord, porcentajes_ord):
    axes[0].barh(0, p, left=pos, height=1, color=c/255.0)
    pos += p
axes[0].set_xlim(0,100); axes[0].set_title('Paleta dominante (% en imagen)', fontsize=10); axes[0].axis('off')

img_reconstruida = colores_dominantes[etiquetas].reshape(img_rgb.shape)
axes[1].imshow(img_reconstruida)
axes[1].set_title(f'Imagen reconstruida con {N_COLORES} colores', fontsize=10)
axes[1].axis('off')
fig.suptitle('Paleta K-means', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('paleta_kmeans.png', dpi=100, bbox_inches='tight')
plt.show()

color_base = colores_ord[0]
hsv_base = cv2.cvtColor(color_base.reshape(1,1,3), cv2.COLOR_RGB2HSV)[0,0]
H, S, V = int(hsv_base[0]), int(hsv_base[1]), int(hsv_base[2])

def hsv_a_rgb(h, s, v):
    c = np.array([[[h%180, s, v]]], dtype=np.uint8)
    return cv2.cvtColor(c, cv2.COLOR_HSV2RGB)[0,0]

armonias = {
    'Base'           : [hsv_a_rgb(H,S,V)],
    'Complementario' : [hsv_a_rgb(H,S,V), hsv_a_rgb(H+90,S,V)],
    'Analogos'       : [hsv_a_rgb(H-15,S,V), hsv_a_rgb(H,S,V), hsv_a_rgb(H+15,S,V)],
    'Triadico'       : [hsv_a_rgb(H,S,V), hsv_a_rgb(H+60,S,V), hsv_a_rgb(H+120,S,V)],
}

fig, axes = plt.subplots(1, 4, figsize=(12, 2.5))
for ax, (nombre, cols) in zip(axes, armonias.items()):
    ax.imshow(np.array(cols).reshape(1,-1,3))
    ax.set_title(nombre, fontsize=10); ax.axis('off')
fig.suptitle('Armonias cromaticas desde color dominante', fontsize=12, y=1.1)
plt.tight_layout()
plt.savefig('armonias_cromaticas.png', dpi=100, bbox_inches='tight')
plt.show()
print('Paletas y armonias generadas.')

## 📊 Sección 9 — Análisis de histogramas

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, (c, n) in enumerate(zip(['#ff5252','#69f0ae','#448aff'], ['R','G','B'])):
    hist = cv2.calcHist([img_rgb], [i], None, [256], [0,256])
    axes[0,0].plot(hist, color=c, alpha=0.8, label=n, linewidth=1.5)
axes[0,0].set_title('Histograma RGB', fontsize=11)
axes[0,0].legend(); axes[0,0].grid(True)

for i, (c, n) in enumerate(zip(['#ffcc02','#ff7043','#c0c0c0'], ['H','S','V'])):
    hist = cv2.calcHist([img_hsv], [i], None, [256], [0,256])
    axes[0,1].plot(hist, color=c, alpha=0.8, label=n, linewidth=1.5)
axes[0,1].set_title('Histograma HSV', fontsize=11)
axes[0,1].legend(); axes[0,1].grid(True)

gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
gray_eq = cv2.equalizeHist(gray)
axes[0,2].hist(gray.ravel(),    256, color='#6a6f8a', alpha=0.6, label='Original')
axes[0,2].hist(gray_eq.ravel(), 256, color='#4fc3f7', alpha=0.6, label='Ecualizado')
axes[0,2].set_title('Ecualizacion de histograma', fontsize=11)
axes[0,2].legend(); axes[0,2].grid(True)

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
gray_clahe = clahe.apply(gray)
axes[1,0].imshow(gray, cmap='gray');       axes[1,0].set_title('Original (gris)', fontsize=10);    axes[1,0].axis('off')
axes[1,1].imshow(gray_eq, cmap='gray');    axes[1,1].set_title('Ecualizacion global', fontsize=10); axes[1,1].axis('off')
axes[1,2].imshow(gray_clahe, cmap='gray'); axes[1,2].set_title('CLAHE adaptativa', fontsize=10);   axes[1,2].axis('off')

fig.suptitle('Analisis y ecualizacion de histogramas', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('histogramas.png', dpi=100, bbox_inches='tight')
plt.show()
print('Histogramas completados.')

## 🎁 Sección 10 — Bonus: Corrección automática, daltonismo y transfer de color

In [ ]:
def correccion_automatica(img_rgb):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    lab[:,:,0] = clahe.apply(lab[:,:,0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

img_corregida = correccion_automatica(img_rgb)

color_dom = colores_ord[0]
hsv_dom = cv2.cvtColor(color_dom.reshape(1,1,3), cv2.COLOR_RGB2HSV)[0,0]
matices = {(0,15):'Rojo',(15,35):'Naranja',(35,50):'Amarillo',(50,80):'Verde',(80,130):'Azul',(130,160):'Morado',(160,180):'Rojo'}
nombre_dom = next((n for (lo,hi),n in matices.items() if lo<=hsv_dom[0]<hi), 'Indefinido')
print(f'Color dominante: {nombre_dom}  RGB:{tuple(color_dom)}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(img_rgb);       axes[0].set_title('Original', fontsize=11);            axes[0].axis('off')
axes[1].imshow(img_corregida); axes[1].set_title('Correccion automatica (CLAHE LAB)', fontsize=11); axes[1].axis('off')
fig.suptitle('Correccion automatica de color', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('correccion_automatica.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
def simular_daltonismo(img_rgb, tipo):
    matrices = {
        'Protanopia':   np.array([[0.567,0.433,0.000],[0.558,0.442,0.000],[0.000,0.242,0.758]]),
        'Deuteranopia': np.array([[0.625,0.375,0.000],[0.700,0.300,0.000],[0.000,0.300,0.700]]),
        'Tritanopia':   np.array([[0.950,0.050,0.000],[0.000,0.433,0.567],[0.000,0.475,0.525]]),
    }
    M = matrices[tipo]
    img_f = img_rgb.astype(np.float32) / 255.0
    return np.clip(np.dot(img_f, M.T)*255, 0, 255).astype(np.uint8)

tipos = ['Protanopia','Deuteranopia','Tritanopia']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img_rgb); axes[0].set_title('Original', fontsize=10); axes[0].axis('off')
for i, tipo in enumerate(tipos):
    axes[i+1].imshow(simular_daltonismo(img_rgb, tipo))
    axes[i+1].set_title(tipo, fontsize=10); axes[i+1].axis('off')
fig.suptitle('Simulacion de daltonismo', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('simulacion_daltonismo.png', dpi=100, bbox_inches='tight')
plt.show()
print('Simulacion de daltonismo completada.')

In [ ]:
from skimage import color as skcolor

def transfer_color(fuente_rgb, destino_rgb):
    fuente_lab  = skcolor.rgb2lab(fuente_rgb  / 255.0)
    destino_lab = skcolor.rgb2lab(destino_rgb / 255.0)
    resultado_lab = destino_lab.copy()
    for canal in range(3):
        mf = fuente_lab[:,:,canal].mean()
        sf = fuente_lab[:,:,canal].std()
        md = destino_lab[:,:,canal].mean()
        sd = destino_lab[:,:,canal].std()
        if sd > 0:
            resultado_lab[:,:,canal] = (destino_lab[:,:,canal] - md) * (sf/sd) + mf
    return np.clip(skcolor.lab2rgb(resultado_lab)*255, 0, 255).astype(np.uint8)

img_transferida = transfer_color(filtro_warm, filtro_cool)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(filtro_warm);     axes[0].set_title('Fuente (paleta calida)', fontsize=10);     axes[0].axis('off')
axes[1].imshow(filtro_cool);     axes[1].set_title('Destino (paleta fria)',  fontsize=10);      axes[1].axis('off')
axes[2].imshow(img_transferida); axes[2].set_title('Destino + color de fuente', fontsize=10); axes[2].axis('off')
fig.suptitle('Transfer de color (Reinhard et al.)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('transfer_color.png', dpi=100, bbox_inches='tight')
plt.show()
print('Transfer de color completado.')

In [ ]:
import zipfile, os
archivos = [
    'imagen_original.png','canales_rgb.png','canales_hsv.png','canales_lab.png',
    'canales_ycrcb.png','espacios_3d.png','segmentacion_color.png',
    'manipulacion_color.png','color_grading.png','paleta_kmeans.png',
    'armonias_cromaticas.png','histogramas.png','correccion_automatica.png',
    'simulacion_daltonismo.png','transfer_color.png',
]
with zipfile.ZipFile('outputs_color.zip', 'w') as zf:
    for archivo in archivos:
        if os.path.exists(archivo):
            zf.write(archivo); print(f'  OK {archivo}')
        else:
            print(f'  NO ENCONTRADO: {archivo}')
from google.colab import files
files.download('outputs_color.zip')
print('ZIP descargado.')